# Unit 7 — Capstone: a coding agent ⭐

Everything from the previous units, together.

> This notebook walks through the pieces. The interactive REPL lives in `agent.py` — run that in a terminal for the full experience.

In [ ]:
from dotenv import load_dotenv
load_dotenv(dotenv_path="../.env")

import asyncio, subprocess
from collections.abc import Awaitable, Callable
from pathlib import Path
from typing import Annotated, Any
from pydantic import Field
from agent_framework import (
    AgentResponse, FunctionInvocationContext, MCPStdioTool, Message,
    function_middleware, tool,
)
from agent_framework.openai import OpenAIChatCompletionClient

## Sandbox

Everything happens inside `./workspace/`. `_safe_path` rejects any attempt to escape it with `../`.

In [ ]:
WORKSPACE = (Path.cwd() / "workspace").resolve()
WORKSPACE.mkdir(exist_ok=True)

def _safe(p: str) -> Path:
    cand = (WORKSPACE / p).resolve()
    if WORKSPACE not in cand.parents and cand != WORKSPACE:
        raise ValueError(f"{p!r} escapes sandbox")
    return cand

## Tools

Notice `approval_mode`:
- `read_file`, `list_dir` — `"never_require"` (safe, read-only)
- `write_file`, `run_shell` — `"always_require"` (dangerous)

In [ ]:
@tool(approval_mode="never_require")
def read_file(path: Annotated[str, Field(description="Path relative to workspace.")]) -> str:
    """Read a text file from the workspace."""
    p = _safe(path)
    return p.read_text(encoding="utf-8", errors="replace") if p.exists() else f"error: {path} missing"

@tool(approval_mode="never_require")
def list_dir(path: Annotated[str, Field(description="Dir path; '.' for root.")] = ".") -> str:
    """List a directory in the workspace."""
    p = _safe(path)
    return "\n".join(f"{'dir' if e.is_dir() else 'file'}  {e.name}" for e in sorted(p.iterdir()))

@tool(approval_mode="always_require")
def write_file(path: Annotated[str, Field(description="File path.")],
               content: Annotated[str, Field(description="Contents.")]) -> str:
    """Write a file. Requires approval."""
    p = _safe(path); p.parent.mkdir(parents=True, exist_ok=True)
    p.write_text(content, encoding="utf-8")
    return f"wrote {len(content)} bytes to {path}"

@tool(approval_mode="always_require")
def run_shell(command: Annotated[str, Field(description="Shell command.")]) -> str:
    """Run a shell command inside the workspace. Requires approval."""
    r = subprocess.run(command, shell=True, cwd=str(WORKSPACE), capture_output=True, text=True, timeout=60)
    return f"exit={r.returncode}\n{(r.stdout + r.stderr)[-2000:]}"

## Middleware + approval loop

Same patterns as Units 3 and 6.

In [ ]:
def _args_to_dict(args):
    if args is None:
        return {}
    if hasattr(args, "model_dump"):
        return args.model_dump()
    try:
        return dict(args)
    except (TypeError, ValueError):
        return {"_repr": str(args)}

@function_middleware
async def log_tool_calls(ctx: FunctionInvocationContext, call_next: Callable[[], Awaitable[None]]) -> None:
    name = ctx.function.name
    args_dict = _args_to_dict(ctx.arguments)
    args = {k: (str(v)[:60] + "..." if len(str(v)) > 60 else v) for k, v in args_dict.items()}
    print(f"\n  [tool →] {name}({args})")
    await call_next()
    print(f"  [tool ←] {name} → {str(ctx.result)[:200]}")

async def run_verbose(query, agent, session, max_rounds: int = 20):
    """Stream the response, collect approvals, resume until done."""
    current_input = query
    rounds = 0
    while True:
        rounds += 1
        if rounds > max_rounds:
            raise RuntimeError(f"Exceeded {max_rounds} approval rounds")

        user_input_requests = []
        print(f"\n{agent.name}: ", end="", flush=True)
        async for chunk in agent.run(current_input, stream=True, session=session):
            if chunk.text:
                print(chunk.text, end="", flush=True)
            if chunk.user_input_requests:
                user_input_requests.extend(chunk.user_input_requests)
        print()

        if not user_input_requests:
            return

        responses = []
        for r in user_input_requests:
            print(f"\nApproval requested: {r.function_call.name}({r.function_call.arguments})")
            answer = await asyncio.to_thread(input, "Approve? (y/n): ")
            ok = answer.strip().lower().startswith("y")
            responses.append(Message("user", [r.to_function_approval_response(ok)]))
        current_input = responses

## Run a one-shot task (streaming + inline approval)

The agent narrates each step, pauses when it wants to write or run, and resumes after your y/n.

In [ ]:
import contextlib

INSTRUCTIONS = """You are a careful, talkative coding assistant in the workspace/ folder.
Before each tool call, say one short sentence about what you're doing and why.
After each tool returns, say one short sentence about what you learned.
Read before writing. Run tests after writing.
"""

@contextlib.asynccontextmanager
async def optional_fs_mcp():
    """Yield filesystem MCP tool if it starts, else None."""
    try:
        async with MCPStdioTool(
            name="filesystem",
            command="npx",
            args=["-y", "@modelcontextprotocol/server-filesystem", str(WORKSPACE)],
            load_prompts=False,    # server exposes tools only
            tool_name_prefix="fs_", # avoid name collision with our custom tools
        ) as fs:
            print("[mcp] filesystem ready")
            yield fs
    except Exception as e:
        print(f"[mcp] unavailable ({type(e).__name__}): falling back to custom tools")
        yield None

async with optional_fs_mcp() as fs:
    tools = [read_file, list_dir, write_file, run_shell]
    if fs is not None:
        tools.append(fs)
    agent = OpenAIChatCompletionClient().as_agent(
        name="CodingAgent",
        instructions=INSTRUCTIONS,
        tools=tools,
        middleware=[log_tool_calls],
    )
    session = agent.create_session()
    await run_verbose(
        "Create hello.py that prints 'hello from agent', then run it.",
        agent, session,
    )